# PART1 → PART2 방향 전환 결정

**전환 배경**:  
PART1 텍스트 분석(06, 07) 수행 중 두 가지 구조적 문제를 발견했다.

1. **업종 이질성**: 태국음식·스시·버거·피자가 같은 데이터셋에 있어 FPI 구간별 키워드가 "경쟁 환경의 차이"가 아닌 "업종 차이"를 반영하고 있었다.
2. **카지노 노이즈**: 생존/고전 브랜드 비교에서 `hotel, casino, mgm, resort`가 상위 키워드로 등장해 음식·서비스 품질 차이를 가리고 있었다.

**전환 결정**: 패스트푸드 업종으로 범위를 좁혀 재분석.  
- Pizza, Chicken은 배달 중심·저녁 수요 비중이 높아 입지 경쟁 구조가 다르다고 판단하여 제외.
- 단, Yelp 데이터에서 `Fast Food` 카테고리만 적용하면 독립 브랜드 표본이 분석에 충분하지 않았다.
- 이를 보완하기 위해 패스트푸드와 동일한 즉석 서비스(counter service) 방식을 공유하는 `Burgers`, `Sandwiches` 카테고리를 추가 포함하였다.
- 최종 필터: `Burgers + Sandwiches + Fast Food` 3개 카테고리

---
# 버거+샌드위치+패스트푸드 독립 브랜드 FPI 분석
## 01. 데이터 준비 및 프랜차이즈 판정

## 분석 개요

전체 Restaurants에서 아쉬웠던 업종 이질성 문제를 해결하기 위해
버거+샌드위치+패스트푸드 업종으로 범위를 좁혀 동일한 분석을 수행한다.

| 항목 | 팀플 | 사이드 프로젝트 |
|---|---|---|
| 분석 대상 | Las Vegas Restaurants 전체 | Las Vegas 버거+샌드위치+패스트푸드 |
| 독립 브랜드 | 4,818개 | 775개 |
| 분석 목적 | FPI 개념 검증 | 업종 특화 분석 |

**입력 데이터**
- `yelp_business.csv`: 업체 정보
- `yelp_review.csv`: 리뷰 데이터

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BUSINESS_PATH = "../../dataset/yelp_dataset"
REVIEW_PATH   = "../../dataset/yelp_dataset"
PATH_to_save  = "../results"

# 원본 데이터 로드
business_raw = pd.read_csv(f"{BUSINESS_PATH}/yelp_business.csv")
review_raw   = pd.read_csv(f"{REVIEW_PATH}/yelp_review.csv")

print(f"business: {business_raw.shape}")
print(f"review  : {review_raw.shape}")

business: (174567, 13)
review  : (5261668, 9)


### 1-1. 분석 대상 업체 필터링

버거+샌드위치+패스트푸드 카테고리 + Las Vegas 조건으로 필터링한다.

In [3]:
# 버거+샌드위치+패스트푸드 + Las Vegas 필터링
target_cats = ['Burgers', 'Sandwiches', 'Fast Food']

mask = (
    business_raw['city'] == 'Las Vegas'
) & (
    business_raw['categories'].apply(
        lambda x: any(cat in str(x) for cat in target_cats) if pd.notna(x) else False
    )
)

biz_lv = business_raw[mask].copy()

print(f"Las Vegas 버거+샌드위치+패스트푸드: {len(biz_lv):,}개")
print(f"\n업종별 분포 (중복 포함):")
for cat in target_cats:
    n = biz_lv['categories'].str.contains(cat, na=False).sum()
    print(f"  {cat}: {n}개")

Las Vegas 버거+샌드위치+패스트푸드: 1,579개

업종별 분포 (중복 포함):
  Burgers: 531개
  Sandwiches: 655개
  Fast Food: 884개


### 1-2. 프랜차이즈 판정

restaurant과 동일한 기준을 적용한다.
- 브랜드명 10개 이상 등장: 프랜차이즈
- 화이트리스트: 버거+샌드위치+패스트푸드 업종 주요 프랜차이즈 수동 추가

In [12]:
# 브랜드명에서 앞뒤 큰따옴표 제거
biz_lv['name'] = biz_lv['name'].str.strip('"')

# 확인
print("전처리 후 브랜드명 샘플:")
print(biz_lv['name'].head(10).tolist())

# 브랜드명 10개 이상 재계산
name_counts = biz_lv['name'].value_counts()
franchise_by_count = set(name_counts[name_counts >= 10].index)

# 화이트리스트 수정본
whitelist = {
    "McDonald's", "McDonalds",
    "Subway", "Burger King", "Wendy's",
    "Jack in the Box", "Jack In the Box",
    "Carl's Jr.", "Carl's Jr",
    "Five Guys", "In-N-Out Burger",
    "Whataburger", "Sonic Drive-In",
    "Jimmy John's", "Jersey Mike's",
    "Firehouse Subs", "Arby's",
    "Popeyes", "Popeyes Louisiana Kitchen",
    "Chick-fil-A", "Wingstop",
    "Raising Cane's", "Shake Shack",
    "Habit Burger", "Which Wich",
    "Quiznos", "Blimpie", "Charley's",
    "Potbelly", "Jason's Deli",
    "Fatburger", "Wienerschnitzel",
    "Panera Bread", "Checkers",
    "Church's Chicken", "Nathan's Famous",
    "Long John Silver's",
    "Applebee's Neighborhood Grill & Bar"
}

franchise_names = franchise_by_count | whitelist
biz_lv['is_franchise'] = biz_lv['name'].isin(franchise_names)

print(f"\n브랜드명 10개↑: {len(franchise_by_count)}개")
print(f"\n프랜차이즈: {biz_lv['is_franchise'].sum():,}개")
print(f"독립 브랜드: {(~biz_lv['is_franchise']).sum():,}개")

# 다시 확인
indie_check = biz_lv[~biz_lv['is_franchise']].copy()
print("\n독립 브랜드 중 다점포 (5개 이상):")
cnt = indie_check['name'].value_counts()
print(cnt[cnt >= 5])

전처리 후 브랜드명 샘플:
['Subway', "McDonald's", 'Subway', 'Burger King', 'Panda Express', 'Del Mar Deli', "McDonald's", "Tootie's Place", 'The Hummus Factory', 'Table 89']

브랜드명 10개↑: 28개

프랜차이즈: 804개
독립 브랜드: 775개

독립 브랜드 중 다점포 (5개 이상):
name
Tropical Smoothie Cafe    9
Farmer Boys               7
Rachel's Kitchen          5
Name: count, dtype: int64


### 1-3. 분석 대상 저장

In [13]:
# business_raw도 name 전처리 적용
business_raw['name'] = business_raw['name'].str.strip('"')

# 최종 분석 대상
biz_target = biz_lv.copy()
review_target = review_raw[
    review_raw['business_id'].isin(biz_target['business_id'])
].copy()

print(f"분석 대상 업체: {len(biz_target):,}개")
print(f"분석 대상 리뷰: {len(review_target):,}개")
print(f"\n프랜차이즈: {biz_target['is_franchise'].sum():,}개")
print(f"독립 브랜드: {(~biz_target['is_franchise']).sum():,}개")
print(f"영업 중 독립 브랜드: {((~biz_target['is_franchise']) & (biz_target['is_open']==1)).sum():,}개")

# 저장
biz_target.to_csv(f"{PATH_to_save}/biz_target_burger.csv", index=False, encoding='utf-8-sig')
review_target.to_csv(f"{PATH_to_save}/review_target_burger.csv", index=False, encoding='utf-8-sig')

print(f"\n저장 완료:")
print(f"  biz_target_burger.csv")
print(f"  review_target_burger.csv")

분석 대상 업체: 1,579개
분석 대상 리뷰: 178,094개

프랜차이즈: 804개
독립 브랜드: 775개
영업 중 독립 브랜드: 535개

저장 완료:
  biz_target_burger.csv
  review_target_burger.csv
